# scRareBench | High-Level scDML | Dataset 2 — mBDRC renal cortex

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

This Google Colab notebook runs **scDML** on scRareBench Dataset 2 (`mbdrc_renal_cortex`) through the generic **MethodSpec → benchmark_method** API.

scDML remains completely user-side. scRareBench loads Dataset 2, orchestrates independent method seeds, validates latent/cell alignment, evaluates each run, and creates the finalized multi-seed delivery.

### Dataset 2 / scDML contract

- Dataset selector: `2` (`mbdrc_renal_cortex`)
- Benchmark label: package-registered Dataset 2 label (`cell_type`) — **never supplied to scDML**
- Integration/evaluation batch: package-registered combined `donor_id × assay`
- Dataset 2 outer feature policy: **global Seurat-v3 4000 HVGs from raw counts**
- Native scDML feature step: **2000 HVGs** selected inside the audited 4000-HVG feature universe
- Initial clustering: Leiden, resolution `3.0`
- scDML normalization: normalize total `1e4`, then log1p
- scDML scaling: independently within each integration batch, `max_value=10`
- PCA: `50`
- Integration: fully unsupervised, `expect_num_cluster=None`
- Merge candidates: `2` through the observed number of initial Leiden clusters
- `K_in=5`, `K_bw=10`, cosine neighbor metrics, merge rule `rule2`
- 50 training epochs, hard triplets, Euclidean metric, margin `0.2`
- Output dimension `32`, hidden embedding layer `[256]`
- Method seeds: `42, 123, 2026`
- Benchmark seed fixed at `42`

The two-stage feature policy is intentional. Dataset 2 first applies its fixed cross-method 4000-HVG policy from raw counts. scDML then performs its native 2000-HVG selection on that feature universe, preserving the method settings used in the previously executed scDML workflow.

> Use a **GPU + High-RAM Colab runtime**. scDML materializes dense matrices during parts of preprocessing/training. For a smoke test, temporarily use `METHOD_SEEDS = [42]`.

## 1. Install scRareBench and scDML

scDML is an older package. The notebook installs its runtime dependencies explicitly and then installs scDML itself with `--no-deps`, matching the modern-Colab pattern used previously. The upstream source is pinned to a fixed commit for reproducibility.


In [ ]:
import subprocess
import sys

SCRAREBENCH_GITHUB = (
    "git+https://github.com/amirhossein-alishahi/scRareBench_.git@cfdb720f5488b9ceef9a225b35fc7558bcb0251c"
)
SCDML_COMMIT = "d762dd75557c0dbb2d3039174d69a18195842b93"
SCDML_GITHUB = f"git+https://github.com/eleozzr/scDML.git@{SCDML_COMMIT}"

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps", SCRAREBENCH_GITHUB
])

METHOD_RUNTIME_DEPENDENCIES = (
    "tables>=3.6.1",
    "annoy>=1.17.0",
    "hnswlib>=0.5.2",
    "pytorch-metric-learning>=0.9.95",
    "numpy-groupies>=0.9.14",
    "numexpr>=2.7.1",
    "networkx>=2.5",
    "igraph>=0.10.0",
    "leidenalg>=0.10.0",
)

from scrarebench.runtime import print_install_report, setup_runtime

install_report = setup_runtime(
    extra_requirements=METHOD_RUNTIME_DEPENDENCIES,
    extra_imports=(
        "tables", "annoy", "hnswlib", "pytorch_metric_learning",
        "numpy_groupies", "numexpr", "networkx", "igraph", "leidenalg",
    ),
    quiet=False,
)
print_install_report(install_report)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps", SCDML_GITHUB
])

import scDML
from scDML import scDMLModel
print("scDML:", getattr(scDML, "__version__", "installed"))
print("Pinned scDML commit:", SCDML_COMMIT)


In [ ]:
# OPTIONAL runtime diagnostics only.
#
# import sys
# import scanpy as sc
# import torch
# from importlib import metadata
#
# print("Python:", sys.version)
# print("scanpy:", sc.__version__)
# print("torch:", torch.__version__)
# print("CUDA:", torch.cuda.is_available())
# for pkg in ("numpy", "anndata", "pytorch-metric-learning", "hnswlib", "annoy"):
#     try:
#         print(pkg, metadata.version(pkg))
#     except metadata.PackageNotFoundError:
#         print(pkg, "not installed")


## 2. Load Dataset 2 / mBDRC


In [ ]:
from scrarebench import load_dataset, dataset_info

adata = load_dataset(2)
info = dataset_info(adata)

print(adata)
print("\nDataset metadata:")
for key, value in info.items():
    print(f"  {key}: {value}")

if info.get("label_key") is None:
    raise RuntimeError(f"Dataset 2 label metadata is missing: {info}")
if info.get("batch_key") is None:
    raise RuntimeError(f"Dataset 2 batch metadata is missing: {info}")
if info.get("scib_hvg_batch_mode") != "global":
    raise RuntimeError(
        "Dataset 2 must use the audited global-HVG policy. "
        f"Observed metadata: {info}"
    )
if not adata.obs_names.is_unique:
    raise ValueError("Dataset 2 observation names must be unique.")

print("Benchmark label:", info["label_key"])
print("Integration/evaluation batch:", info["batch_key"])
print("Dataset 2 HVG batch mode:", info["scib_hvg_batch_mode"])


## 3. Recover and validate raw counts

Dataset 2 global-HVG selection and scDML preprocessing both require genuine raw counts. The notebook prefers a validated `counts` layer, otherwise tries aligned `raw.X`, and accepts `X` only if it is nonnegative and integer-like.


In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp


def sampled_count_diagnostics(matrix, max_values=100_000, seed=42):
    values = matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()
    if len(values) > max_values:
        rng = np.random.default_rng(int(seed))
        values = values[rng.choice(len(values), size=max_values, replace=False)]
    values = np.asarray(values, dtype=float)
    finite_mask = np.isfinite(values)
    finite = values[finite_mask]
    return {
        "finite_fraction": float(np.mean(finite_mask)) if len(values) else 1.0,
        "minimum": float(np.min(finite)) if len(finite) else 0.0,
        "nonnegative": bool(len(finite) == 0 or np.min(finite) >= 0),
        "integer_like": bool(
            len(finite) == 0 or np.allclose(finite, np.rint(finite), atol=1e-6)
        ),
    }


def copy_matrix(matrix):
    return matrix.copy().tocsr() if sp.issparse(matrix) else np.asarray(matrix).copy()


def ensure_counts_layer(adata, layer_key="counts"):
    if layer_key in adata.layers:
        diag = sampled_count_diagnostics(adata.layers[layer_key])
        if diag["nonnegative"] and diag["integer_like"]:
            return f"layers/{layer_key}", diag
        raise ValueError(f"Existing layer {layer_key!r} is not valid raw counts: {diag}")

    if adata.raw is not None:
        raw_names = pd.Index(adata.raw.var_names.astype(str))
        current_names = pd.Index(adata.var_names.astype(str))
        positions = raw_names.get_indexer(current_names)
        if np.all(positions >= 0):
            matrix = adata.raw.X[:, positions]
            diag = sampled_count_diagnostics(matrix)
            if diag["nonnegative"] and diag["integer_like"]:
                adata.layers[layer_key] = copy_matrix(matrix)
                return "raw.X aligned to adata.var_names", diag

    diag = sampled_count_diagnostics(adata.X)
    if diag["nonnegative"] and diag["integer_like"]:
        adata.layers[layer_key] = copy_matrix(adata.X)
        return "X", diag

    raise ValueError(
        "Could not locate nonnegative integer-like raw counts for Dataset 2. "
        f"layers={list(adata.layers)}, raw_present={adata.raw is not None}, "
        f"X_diagnostics={diag}"
    )


COUNT_LAYER = info.get("count_layer") or "counts"
count_source, count_diagnostics = ensure_counts_layer(adata, COUNT_LAYER)

print("Raw-count layer:", COUNT_LAYER)
print("Raw-count source:", count_source)
print("Count diagnostics:", count_diagnostics)

if count_diagnostics["finite_fraction"] != 1.0:
    raise ValueError("Raw counts contain NaN/Inf.")
if not count_diagnostics["nonnegative"]:
    raise ValueError("Raw counts contain negative values.")
if not count_diagnostics["integer_like"]:
    raise ValueError("Raw counts are not integer-like.")


## 4. Reproducibility and configuration

The method seed changes between runs while the benchmark seed remains fixed. Dataset 2 contributes the outer **4000 global Seurat-v3 HVG** policy; scDML keeps its native **2000-HVG** preprocessing inside that feature universe.


In [ ]:
METHOD_SEEDS = [42, 123, 2026]
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    # Dataset 2 outer feature policy.
    "dataset2_n_hvg": 4000,
    "dataset2_hvg_span": 0.3,
    "dataset2_hvg_policy": "global_seurat_v3_raw_counts",

    # Native scDML preprocessing preserved from the previous executed workflow.
    "cluster_method": "leiden",
    "resolution": 3.0,
    "n_high_var": 2000,
    "normalize_samples": True,
    "target_sum": 1e4,
    "log_normalize": True,
    "normalize_features": True,
    "pca_dim": 50,
    "scale_value": 10.0,

    # Native scDML integration/training.
    "K_in": 5,
    "K_bw": 10,
    "K_in_metric": "cosine",
    "K_bw_metric": "cosine",
    "merge_rule": "rule2",
    "num_epochs": 50,
    "projection": False,
    "early_stop": False,
    "batch_size": 64,
    "metric": "euclidean",
    "margin": 0.2,
    "triplet_type": "hard",
    "device": None,
    "out_dim": 32,
    "emb_dim": [256],
    "save_model": False,
}

OUTPUT_DIR = "/content/scrarebench_scdml_dataset2_highlevel"
METHOD_WORK_DIR = "/content/scrarebench_scdml_dataset2_method_runs"


## 5. User-owned scDML runner

Dataset 2 global 4000-HVG selection changes features only. scDML later performs batch-wise concatenation during scaling, which can reorder cells and alter observation names. A private barcode guard is therefore carried through preprocessing, and the final `X_emb` latent is explicitly restored to the original Dataset 2 order. The benchmark label is never supplied to the method.


In [ ]:
from pathlib import Path
import gc
import random

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch

from scDML import scDMLModel
from scrarebench import MethodOutput

_ORIGINAL_BARCODE_COL = "_scrarebench_original_barcode"


def _looks_like_raw_counts(matrix, max_values=100_000, seed=42):
    values = matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()
    values = np.asarray(values)
    if len(values) > max_values:
        rng = np.random.default_rng(int(seed))
        values = values[rng.choice(len(values), size=max_values, replace=False)]
    values = values.astype(float, copy=False)
    if len(values) == 0:
        return True
    if not np.isfinite(values).all():
        return False
    return bool(values.min() >= 0 and np.allclose(values, np.rint(values), atol=1e-6))


def run_scdml(method_adata, seed, config):
    batch_key = info["batch_key"]
    count_layer = info.get("count_layer") or "counts"

    original_order = method_adata.obs_names.astype(str).to_numpy(copy=True)
    if len(np.unique(original_order)) != len(original_order):
        raise ValueError("Dataset 2 observation names must be unique.")

    method_adata.obs[_ORIGINAL_BARCODE_COL] = original_order

    # Keep GEX only if a multimodal feature annotation is present.
    if "feature_types" in method_adata.var.columns:
        ft = method_adata.var["feature_types"].astype(str).str.upper()
        rna_mask = ft.eq("GEX").to_numpy()
        if int(rna_mask.sum()) == 0:
            raise ValueError("No GEX features were found.")
        if int(rna_mask.sum()) != method_adata.n_vars:
            method_adata = method_adata[:, rna_mask].copy()

    method_adata.var_names_make_unique()

    if count_layer not in method_adata.layers:
        raise KeyError(
            f"Raw-count layer {count_layer!r} is required for scDML. "
            f"Available layers: {list(method_adata.layers)}"
        )

    counts = method_adata.layers[count_layer]
    if not _looks_like_raw_counts(counts, seed=seed):
        raise ValueError(f"adata.layers[{count_layer!r}] does not look like raw counts.")

    # Dataset 2 fixed cross-method policy: global Seurat-v3 4000 HVGs.
    # IMPORTANT: no batch_key is supplied to this HVG call.
    sc.pp.highly_variable_genes(
        method_adata,
        layer=count_layer,
        flavor="seurat_v3",
        n_top_genes=min(int(config["dataset2_n_hvg"]), method_adata.n_vars),
        span=float(config["dataset2_hvg_span"]),
        subset=False,
        check_values=True,
    )

    global_hvg_mask = method_adata.var["highly_variable"].fillna(False).to_numpy(dtype=bool)
    expected_global_hvg = min(int(config["dataset2_n_hvg"]), method_adata.n_vars)
    observed_global_hvg = int(global_hvg_mask.sum())
    if observed_global_hvg != expected_global_hvg:
        raise RuntimeError(
            f"Dataset 2 global-HVG selection returned {observed_global_hvg}; "
            f"expected {expected_global_hvg}."
        )

    method_adata = method_adata[:, global_hvg_mask].copy()
    if not np.array_equal(
        method_adata.obs[_ORIGINAL_BARCODE_COL].astype(str).to_numpy(),
        original_order,
    ):
        raise RuntimeError("Cell order changed during Dataset 2 global-HVG subsetting.")

    # scDML.preprocess() consumes adata.X. Use raw counts from the 4000-HVG universe.
    method_adata.X = method_adata.layers[count_layer].copy()

    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

    seed_dir = Path(METHOD_WORK_DIR) / f"seed_{int(seed)}"
    seed_dir.mkdir(parents=True, exist_ok=True)

    model = scDMLModel(verbose=True, save_dir=str(seed_dir) + "/")

    # Native scDML preprocessing: original 2000-HVG selection inside Dataset 2's 4000 HVGs.
    adata_scdml = model.preprocess(
        method_adata,
        cluster_method=str(config["cluster_method"]),
        resolution=float(config["resolution"]),
        batch_key=batch_key,
        n_high_var=int(config["n_high_var"]),
        normalize_samples=bool(config["normalize_samples"]),
        target_sum=float(config["target_sum"]),
        log_normalize=bool(config["log_normalize"]),
        normalize_features=bool(config["normalize_features"]),
        pca_dim=int(config["pca_dim"]),
        scale_value=float(config["scale_value"]),
        mode="unsupervised",
    )

    if _ORIGINAL_BARCODE_COL not in adata_scdml.obs:
        raise RuntimeError("The original-barcode guard was lost during scDML preprocessing.")

    num_init_clusters = int(adata_scdml.obs["init_cluster"].nunique())
    if num_init_clusters < 2:
        raise RuntimeError(f"Too few initial clusters: {num_init_clusters}")

    ncluster_candidates = list(range(2, num_init_clusters + 1))
    print(
        f"Seed {seed}: Dataset2 global HVGs={observed_global_hvg}; "
        f"initial Leiden clusters={num_init_clusters}; "
        f"merge candidates=2..{num_init_clusters}"
    )

    model.integrate(
        adata_scdml,
        batch_key=batch_key,
        ncluster_list=ncluster_candidates,
        expect_num_cluster=None,
        K_in=int(config["K_in"]),
        K_bw=int(config["K_bw"]),
        K_in_metric=str(config["K_in_metric"]),
        K_bw_metric=str(config["K_bw_metric"]),
        merge_rule=str(config["merge_rule"]),
        num_epochs=int(config["num_epochs"]),
        projection=bool(config["projection"]),
        early_stop=bool(config["early_stop"]),
        batch_size=int(config["batch_size"]),
        metric=str(config["metric"]),
        margin=float(config["margin"]),
        triplet_type=str(config["triplet_type"]),
        device=config["device"],
        seed=int(seed),
        out_dim=int(config["out_dim"]),
        emb_dim=list(config["emb_dim"]),
        save_model=bool(config["save_model"]),
        celltype_key=None,
        mode="unsupervised",
    )

    if "X_emb" not in adata_scdml.obsm:
        raise RuntimeError("scDML did not create adata.obsm['X_emb'].")

    latent_native = np.asarray(adata_scdml.obsm["X_emb"], dtype=np.float32)
    native_barcodes = adata_scdml.obs[_ORIGINAL_BARCODE_COL].astype(str).to_numpy()

    if latent_native.ndim != 2 or latent_native.shape[0] != len(native_barcodes):
        raise RuntimeError(f"Unexpected scDML latent shape: {latent_native.shape}")
    if latent_native.shape[1] != int(config["out_dim"]):
        raise RuntimeError(
            f"Unexpected scDML latent width {latent_native.shape[1]}; "
            f"expected {config['out_dim']}."
        )
    if not np.isfinite(latent_native).all():
        raise ValueError("scDML latent contains NaN/Inf.")
    if len(np.unique(native_barcodes)) != len(native_barcodes):
        raise RuntimeError("Original barcodes are no longer unique after preprocessing.")
    if set(native_barcodes) != set(original_order):
        raise RuntimeError("scDML preprocessing changed the Dataset 2 cell set.")

    position = pd.Series(np.arange(len(native_barcodes)), index=native_barcodes)
    reorder = position.loc[original_order].to_numpy()
    latent = latent_native[reorder]

    if not np.array_equal(native_barcodes[reorder], original_order):
        raise RuntimeError("Failed to restore scDML latent to original Dataset 2 order.")

    provenance = {}
    log_path = seed_dir / "log.txt"
    if log_path.exists():
        provenance["scdml_log"] = log_path

    print(f"Seed {seed}: aligned latent shape={latent.shape}")

    del model, adata_scdml, latent_native, position, reorder
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return MethodOutput(
        latent=latent,
        barcodes=original_order,
        representation_key="X_scdml",
        provenance_files=provenance,
    )


## 6. Run the High-Level multi-seed benchmark

For the first compatibility run, temporarily change `METHOD_SEEDS` to `[42]`. Restore all three seeds for the final benchmark.


In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="scDML",
    runner=run_scdml,
    config=METHOD_CONFIG,
    dependencies=METHOD_RUNTIME_DEPENDENCIES + (SCDML_GITHUB,),
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    output_dir=OUTPUT_DIR,
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        "run_scib": True,
        "scib_n_hvg": 4000,
        "scib_reference_n_pcs": 50,
        "scib_n_jobs": 1,
        "scib_progress_bar": True,
        "scib_include_silhouette_batch": True,
        "scib_require_success": True,
        "include_latent_in_bundle": False,
    },
    install_dependencies=False,
    finalize=True,
)


## 7. Inspect the finalized multi-seed result


In [ ]:
from IPython.display import display

display(result.summary().round(5))
print("Method:", result.method)
print("Completed seeds:", result.seeds)
print("Multi-seed report:", result.report_path)
print("Delivery archive:", result.archive_path)
print("Summary JSON:", result.summary_path)

for seed, run in result.runs.items():
    print(f"\nSeed {seed}")
    print("  representation:", run.representation_key)
    print("  output:", run.output_dir)
    print("  interactive report:", run.interactive_report_path)
    print("  PDF:", run.pdf_path)
    print("  bundle:", run.bundle_path)


## 8. Download the final multi-seed archive


In [ ]:
from pathlib import Path
from google.colab import files

archive_path = Path(result.archive_path)
if not archive_path.exists():
    raise FileNotFoundError(
        f"Multi-seed output archive was not found: {archive_path}\n"
        "Make sure benchmark_method(..., finalize=True) completed successfully."
    )

print(f"Downloading: {archive_path.name}")
print(f"Size: {archive_path.stat().st_size / (1024**2):.2f} MB")
files.download(str(archive_path))
